In [63]:
%reset -f

In [64]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, confusion_matrix, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.semi_supervised import LabelSpreading
from sklearn.manifold import SpectralEmbedding
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

In [65]:
data = pd.read_parquet(Path.cwd().parent.parent / 'TabulatedData' / 'values_v6_32bit.parquet')

In [66]:
label = "Jiya Sachdeva"

# Create Balanced Binary Classification Dataset

In [67]:
# Get data for the specific person
person_data = data[data['name'] == label]
n_person_samples = len(person_data)

print(f"Found {n_person_samples} samples for {label}")

# Get data for all other people
other_data = data[data['name'] != label]

# Sample equal number of data points from other people
# If there are multiple people, sample proportionally or randomly
if len(other_data) >= n_person_samples:
    other_sampled = other_data.sample(n=n_person_samples, random_state=42)
else:
    print(f"Warning: Only {len(other_data)} samples available from other people. Using all available samples.")
    other_sampled = other_data
    # Adjust person samples to match
    person_data = person_data.sample(n=len(other_sampled), random_state=42)
    n_person_samples = len(person_data)

print(f"Using {n_person_samples} samples from {label}")
print(f"Using {len(other_sampled)} samples from other people")

# Combine the datasets
person_data['target'] = 1  # Positive class (the specific person)
other_sampled['target'] = 0  # Negative class (other people)

balanced_data = pd.concat([person_data, other_sampled], ignore_index=True)

# Shuffle the data
balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset created with {len(balanced_data)} total samples")
print(f"Class distribution: {balanced_data['target'].value_counts().to_dict()}")

Found 254 samples for Jiya Sachdeva
Using 254 samples from Jiya Sachdeva
Using 254 samples from other people

Balanced dataset created with 508 total samples
Class distribution: {1: 254, 0: 254}


# Prepare Features and Target

In [68]:
# Features (drop name and target columns)
X = balanced_data.drop(['name', 'target'], axis=1)
y = balanced_data['target']

del balanced_data, data, person_data, other_data, other_sampled
print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

Feature matrix shape: (508, 6096)
Target distribution: {1: 254, 0: 254}


# Split Data and Scale Features

In [69]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
del X, y

Training set: (406, 6096)
Test set: (102, 6096)


# Define and Train Multiple Models

The algorithms implemented are
- Random Forests
- Support Vector Machine
- Logistic Regression
- Decision Tree
- XGBoost
- Naive Bayes - Gaussian NB
- K-Nearest Neighbours
- Spectral Embedding + Support Vector Machine

In [70]:
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(objective='binary:logistic', random_state=42, tree_method='hist', n_jobs = -1),
    'Naive Bayes-GaussianNB': GaussianNB(),
    'KNN': KNeighborsClassifier(),
    'MLP': MLPClassifier(hidden_layer_sizes=(50,), solver='adam', activation='relu', random_state=42),
}

results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")

    try:
        # Handle semi-supervised models
        if name == 'LabelSpreading':
            # LabelSpreading works directly on features
            model.fit(X_train_scaled, y_train)
            y_pred_train = model.predict(X_train_scaled)
            y_pred_test = model.predict(X_test_scaled)
            X_test_for_proba = X_test_scaled

        elif name == 'Spectral Embedding + SVM':
            # Spectral Embedding for dimensionality reduction
            embedder = SpectralEmbedding(
                n_components=50,
                affinity='nearest_neighbors',
                n_neighbors=10,
                random_state=42
            )

            X_train_embed = embedder.fit_transform(X_train_scaled)
            X_test_embed = embedder.fit_transform(X_test_scaled)  # Note: Spectral Embedding doesn't have transform, so we use fit_transform

            # Train SVM on embedded space
            model.fit(X_train_embed, y_train)
            y_pred_train = model.predict(X_train_embed)
            y_pred_test = model.predict(X_test_embed)
            X_test_for_proba = X_test_embed

        else:
            # Standard supervised models
            model.fit(X_train_scaled, y_train)
            y_pred_train = model.predict(X_train_scaled)
            y_pred_test = model.predict(X_test_scaled)
            X_test_for_proba = X_test_scaled

        # Calculate metrics
        train_acc = accuracy_score(y_train, y_pred_train)
        test_acc = accuracy_score(y_test, y_pred_test)
        f1 = f1_score(y_test, y_pred_test, average='weighted')
        precision = precision_score(y_test, y_pred_test)
        recall = recall_score(y_test, y_pred_test)

        # Calculate FAR and FRR from confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
        far = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Acceptance Rate
        frr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Rejection Rate

        # Calculate EER (Equal Error Rate)
        # EER is the threshold where FAR = FRR
        try:
            # Get probabilities for EER calculation (only for models that support predict_proba)
            if hasattr(model, 'predict_proba'):
                y_proba = model.predict_proba(X_test_for_proba)[:, 1]
            elif hasattr(model, 'decision_function'):
                y_proba = model.decision_function(X_test_for_proba)
            else:
                y_proba = y_pred_test.astype(float)

            fpr, fnr, _ = roc_curve(y_test, y_proba)
            # FNR = 1 - TPR
            fnr = 1 - (tp / (tp + fn)) if (tp + fn) > 0 else 1
            # Find EER (point where FPR ≈ FNR)
            eer = np.min(np.abs(fpr - fnr))
        except Exception as eer_error:
            eer = np.abs(far - frr)  # Fallback to absolute difference

        results[name] = {
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'f1_score': f1,
            'precision': precision,
            'recall': recall,
            'far': far,
            'frr': frr,
            'eer': eer,
            'model': model
        }

        print(f"{name} Results:")
        print(f"Training Accuracy: {train_acc:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")

        # Print classification report
        print("\nClassification Report (Test Set):")
        print(classification_report(y_test, y_pred_test, target_names=[f'Not {label}', label]))

    except Exception as e:
        print(f"Error training {name}: {str(e)}")
        results[name] = {'error': str(e)}



Training Random Forest...
Random Forest Results:
Training Accuracy: 1.0000
Test Accuracy: 0.9510

Classification Report (Test Set):
                   precision    recall  f1-score   support

Not Jiya Sachdeva       0.96      0.94      0.95        51
    Jiya Sachdeva       0.94      0.96      0.95        51

         accuracy                           0.95       102
        macro avg       0.95      0.95      0.95       102
     weighted avg       0.95      0.95      0.95       102


Training SVM...
SVM Results:
Training Accuracy: 0.9754
Test Accuracy: 0.8922

Classification Report (Test Set):
                   precision    recall  f1-score   support

Not Jiya Sachdeva       0.86      0.94      0.90        51
    Jiya Sachdeva       0.93      0.84      0.89        51

         accuracy                           0.89       102
        macro avg       0.90      0.89      0.89       102
     weighted avg       0.90      0.89      0.89       102


Training Logistic Regression...
Logisti

# Summary Results

In [71]:
print("\n" + "="*80)
print(f"SUMMARY OF RESULTS for {label}")
print("="*80)

summary_data = []
for name, result in results.items():
    if 'error' not in result:
        summary_data.append({
            'Model': name,
            'Train Acc': f"{result['train_accuracy']:.4f}",
            'Test Acc': f"{result['test_accuracy']:.4f}",
            'F1 Score': f"{result['f1_score']:.4f}",
            'Precision': f"{result['precision']:.4f}",
            'Recall': f"{result['recall']:.4f}",
            'FAR': f"{result['far']:.4f}",
            'FRR': f"{result['frr']:.4f}",
            'EER': f"{result['eer']:.4f}"
        })
    else:
        summary_data.append({
            'Model': name,
            'Train Acc': 'Error',
            'Test Acc': result['error']
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Find best performing model
valid_results = {k: v for k, v in results.items() if 'error' not in v}
if valid_results:
    best_model = max(valid_results.items(), key=lambda x: x[1]['test_accuracy'])
    print(f"\n{'='*50}")
    print(f"Best Performing Model: {best_model[0]}")
    print(f"Test Accuracy: {best_model[1]['test_accuracy']:.4f}")
    print(f"F1 Score: {best_model[1]['f1_score']:.4f}")
    print(f"Precision: {best_model[1]['precision']:.4f}")
    print(f"Recall: {best_model[1]['recall']:.4f}")
    print(f"FAR (False Acceptance Rate): {best_model[1]['far']:.4f}")
    print(f"FRR (False Rejection Rate): {best_model[1]['frr']:.4f}")
    print(f"EER (Equal Error Rate): {best_model[1]['eer']:.4f}")
    print(f"{'='*50}")


SUMMARY OF RESULTS for Jiya Sachdeva
                 Model Train Acc Test Acc F1 Score Precision Recall    FAR    FRR    EER
         Random Forest    1.0000   0.9510   0.9510    0.9423 0.9608 0.0588 0.0392 0.0000
                   SVM    0.9754   0.8922   0.8919    0.9348 0.8431 0.0588 0.1569 0.0000
   Logistic Regression    1.0000   0.9412   0.9412    0.9412 0.9412 0.0588 0.0588 0.0000
         Decision Tree    1.0000   0.8431   0.8422    0.9070 0.7647 0.0784 0.2353 0.1569
               XGBoost    1.0000   0.9314   0.9314    0.9400 0.9216 0.0588 0.0784 0.0196
Naive Bayes-GaussianNB    0.9039   0.7745   0.7745    0.7692 0.7843 0.2353 0.2157 0.0196
                   KNN    0.9015   0.8824   0.8823    0.8980 0.8627 0.0980 0.1373 0.0392
                   MLP    1.0000   0.9020   0.9020    0.9020 0.9020 0.0980 0.0980 0.0000

Best Performing Model: Random Forest
Test Accuracy: 0.9510
F1 Score: 0.9510
Precision: 0.9423
Recall: 0.9608
FAR (False Acceptance Rate): 0.0588
FRR (False Reje